# 04 - Let's build a synthetic control

**Business Loves Uplift: Causal Impact Evaluation With Python**<br>
ODSC AI West 2026 · Alex Molak · [CausalPython.io](https://causalpython.io)

<a href="https://colab.research.google.com/github/AlxndrMlk/talks-and-workshops/blob/main/2026-10--odsc-ai/04%20-%20Synthetic%20control.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>


**The story.** A retailer's field sales reps each look after 120-200 stores. The data team built a recommender that tells a rep which new products (SKUs) to list in each store. How much does it lift store sales?

The data is synthetic, so we know the true answer. Weekly sales are in k$ per store.

**The rollout.** The recommender went live in the North in week 76 (week 52 in the table below, which starts at the beginning of the pre-period).

Synthetic control builds a "North without the recommender" as a weighted mix of the other regions, chosen to track the North before the rollout. After the rollout, the gap between the North and its synthetic twin is the effect.

In the second part, we look at a world where this breaks, and at what to do about it.

In [ ]:
# Colab only: install the packages and get the data generator (locally, use the workshop env)
import sys
if "google.colab" in sys.modules:
    !pip install -q dowhy==0.14 econml==0.17.0 cluster-experiments==0.31.0 mlsynth==1.0.0
    !wget -q -nc https://raw.githubusercontent.com/AlxndrMlk/talks-and-workshops/main/2026-10--odsc-ai/uplift_data.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.spatial import ConvexHull
from mlsynth import VanillaSC, SDID

import uplift_data as ud

BLUE, GOLD, GREY, RED, DARK = "#0057d9", "#c8860d", "#bbbbbb", "#c0392b", "#222222"
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
d = ud.make_datasets()     # seed 23: everyone gets the same world
truth = d["truth"]         # the answer key; we only peek at it at the end of each section
print(ud.fingerprint(d))   # should print 13302.553/5496.520

## The data

`region_week`: mean weekly sales per store, by region. 52 weeks before the rollout, 20 after.

In [ ]:
rw = d["region_week"]
N_PRE = 52
rw.head()

In [ ]:
panel = rw.pivot(index="week", columns="region", values="sales")
plt.plot(panel.drop(columns="North"), color=GREY, lw=1)
plt.plot(panel.North, color=BLUE, lw=2, label="North")
plt.axvline(N_PRE, color=DARK, ls="--", lw=1)
plt.text(N_PRE + 0.5, panel.values.max(), "rollout", va="top")
plt.xlabel("week")
plt.ylabel("sales per store (k$)")
plt.legend(frameon=False)
plt.show()

## Fit the synthetic control

`mlsynth`'s `VanillaSC` is the classic method: donor weights are non-negative and sum to one, chosen to fit the North's pre-period sales.

In [ ]:
def fit_sc(df):
    return VanillaSC({"df": df, "outcome": "sales", "treat": "treated", "unitid": "region",
                      "time": "week", "display_graphs": False}).fit()


sc = fit_sc(rw)
pd.Series(sc.donor_weights).sort_values(ascending=False).round(3)

In [ ]:
def plot_sc(panel, synthetic, title, n_pre=N_PRE, fake=None):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    a1.plot(panel.drop(columns="North"), color=GREY, lw=0.8)
    a1.plot(panel.index, panel.North, color=BLUE, lw=2, label="North")
    a1.plot(panel.index, synthetic, color=DARK, ls="--", lw=1.5, label="synthetic North")
    a1.set_ylabel("sales per store (k$)")
    a1.legend(frameon=False)
    a2.plot(panel.index, panel.North - synthetic, color=BLUE)
    a2.axhline(0, color=GREY)
    a2.set_ylabel("gap: North - synthetic (k$)")
    for a in (a1, a2):
        a.axvline(n_pre, color=DARK, ls="--", lw=1)
        if fake is not None:
            a.axvline(fake, color=GOLD, ls=":", lw=1.5)
        a.set_xlabel("week")
    fig.suptitle(title)
    plt.show()


plot_sc(panel, sc.counterfactual, "Synthetic control: the North and its synthetic twin")

In [ ]:
synthetic_post = np.asarray(sc.counterfactual)[N_PRE:].mean()
print(f"pre-period RMSE: {sc.pre_rmse:.2f} k$ (sales are around {panel.North[:N_PRE].mean():.0f} k$)")
print(f"ATT: {sc.att:.2f} k$ per store-week = {sc.att / synthetic_post:.1%}")
print(f"in-space placebo p-value: {sc.inference.p_value:.3f}  (with 8 regions, 1/8 is the best we can get)")

## A confidence interval

The placebo test gives a p-value, not an interval, and with 8 regions it can't go below 1/8. `VanillaSC` has other inference methods that do give an interval for the ATT:

- `"ttest"`: a t-test on cross-fitted SC estimates (Chernozhukov, Wüthrich and Zhu). Fast.
- `"conformal"`: inverts a permutation test on the post-period gaps. Slower (about 15 seconds).

In [ ]:
def sc_ci(df, method):
    """95% interval for the ATT from VanillaSC's `inference` option."""
    return VanillaSC({"df": df, "outcome": "sales", "treat": "treated", "unitid": "region",
                      "time": "week", "display_graphs": False, "inference": method}).fit().att_ci


for method in ["ttest", "conformal"]:
    lo, hi = sc_ci(rw, method)
    print(f"{method:9s} 95% CI: [{lo:.2f}, {hi:.2f}] k$ = [{lo / synthetic_post:.1%}, {hi / synthetic_post:.1%}]")

## Answer key

In [ ]:
lo, hi = sc_ci(rw, "ttest")
print(f"true ATT, North stores: {truth['att_north_store_abs']:.2f} k$ = {truth['att_north_store_pct']:.1%}")
print(f"synthetic control:      {sc.att:.2f} k$ = {sc.att / synthetic_post:.1%}  [{lo:.2f}, {hi:.2f}]")

Close, and the interval covers the truth. Note what SC estimates: the effect on the North (the ATT), averaged over **stores**. DML in notebook 03 averaged over **reps**. Both differ from the ATE over all reps.

---

# Part 2: when synthetic control breaks

The synthetic North is a weighted *average* of donors. An average can't be higher than its highest donor. If the North grows faster than every other region, no mix of donors can follow it: the North sits outside the donors' **convex hull**.

`ud.HULL_SEED` is a world built the same way, where that happens. Let's look at it the way an analyst would.

In [ ]:
d2 = ud.make_datasets(ud.Params(seed=ud.HULL_SEED))
truth2 = d2["truth"]
rw2 = d2["region_week"]
panel2 = rw2.pivot(index="week", columns="region", values="sales")

In [ ]:
def level_and_growth(panel, n_pre=N_PRE):
    """Pre-period mean level and linear growth (k$ per year) of each region."""
    pre = panel.iloc[:n_pre]
    growth = pre.apply(lambda s: np.polyfit(np.arange(n_pre), s, 1)[0] * 52)
    return pd.DataFrame({"level": pre.mean(), "growth": growth})


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, p, name in [(axes[0], panel, "the main world"), (axes[1], panel2, "the HULL_SEED world")]:
    lg = level_and_growth(p)
    donors = lg.drop("North")
    hull = ConvexHull(donors.values)
    loop = list(hull.vertices) + [hull.vertices[0]]
    ax.fill(donors.level.iloc[loop], donors.growth.iloc[loop], color=GREY, alpha=0.3, label="donors' convex hull")
    ax.scatter(donors.level, donors.growth, color=GREY)
    for r, row in donors.iterrows():
        ax.annotate(r, (row.level, row.growth), fontsize=8, xytext=(4, 2), textcoords="offset points")
    ax.scatter(lg.level["North"], lg.growth["North"], color=BLUE, s=90, label="North")
    ax.set_xlabel("pre-period level (k$)")
    ax.set_title(name)
axes[0].set_ylabel("pre-period growth (k$ per year)")
axes[0].legend(frameon=False, loc="lower right")
plt.show()

In the second world, the North grows faster than any donor. Let's fit SC anyway.

In [ ]:
sc2 = fit_sc(rw2)
plot_sc(panel2, sc2.counterfactual, "HULL_SEED world: classic synthetic control")
print(f"pre-period RMSE: {sc2.pre_rmse:.2f} k$")

The pre-period fit looks fine at a glance. But look at the gap before the rollout: it drifts from negative to positive. SC matches the level on average and can't match the slope, so after the rollout the North's extra growth is counted as effect.

**The check: a fake rollout date.** Pretend the rollout happened 20 weeks earlier, and fit only on the weeks before that. There's no effect yet, so no gap should open.

In [ ]:
def fake_date(rw, weeks_before=20):
    """Rollout moved into the pre-period; only pre-period weeks are used."""
    fake = N_PRE - weeks_before
    df = rw[rw.week < N_PRE].copy()
    df["treated"] = ((df.region == "North") & (df.week >= fake)).astype(int)
    return df, fake


df_fake, fake = fake_date(rw2)
sc2_fake = fit_sc(df_fake)
plot_sc(panel2.iloc[:N_PRE], sc2_fake.counterfactual, "HULL_SEED world: fake rollout date", n_pre=N_PRE, fake=fake)
print(f"gap after the fake date: {sc2_fake.att:.2f} k$ ({sc2_fake.att / truth2['att_north_store_abs']:.0%} of the true effect)")

A gap opens where there should be none. The check caught it.

Would a confidence interval have warned us? No. An interval measures noise around the estimate, not bias:

In [ ]:
lo2, hi2 = sc_ci(rw2, "conformal")
print(f"classic SC: {sc2.att:.2f} k$, conformal 95% CI [{lo2:.2f}, {hi2:.2f}], truth {truth2['att_north_store_abs']:.2f} k$")

The interval sits around the biased estimate, and the truth only just makes it in at the lower edge. Diagnostics like the fake-date check tell you about bias; intervals don't.

**Does a fancier SC help?** Synthetic difference-in-differences (SDID) also reweights periods and allows a level shift. It still builds the North from donors, so it has the same problem.

In [ ]:
sdid2 = SDID({"df": rw2, "outcome": "sales", "treat": "treated", "unitid": "region",
              "time": "week", "display_graphs": False}).fit()
print(f"true ATT: {truth2['att_north_store_abs']:.2f} k$")
print(f"classic SC: {sc2.att:.2f} k$ ({sc2.att / truth2['att_north_store_abs'] - 1:+.0%})")
print(f"SDID:       {sdid2.effects.att:.2f} k$ ({sdid2.effects.att / truth2['att_north_store_abs'] - 1:+.0%})")

**Another estimator: SC with a trend.** We remove each region's pre-period level and linear trend, then fit the donor weights on what's left. The North may now grow at its own pace; the donors only need to reproduce its week-to-week movements.

This swaps one assumption for another: the North's extra growth would have continued in a straight line without the recommender. Here that's true, because we built the data that way. With real data, it's an assumption to defend, for example with a longer pre-period.

In [ ]:
def sc_with_trend(panel, n_pre):
    """Synthetic control on series with the pre-period level and linear trend removed."""
    t = np.arange(len(panel))
    Z = np.c_[np.ones(len(panel)), t]
    detrend = lambda s: s - Z @ np.linalg.lstsq(Z[:n_pre], s[:n_pre], rcond=None)[0]
    y = detrend(panel.North.to_numpy())
    donors = panel.drop(columns="North")
    D = np.column_stack([detrend(donors[c].to_numpy()) for c in donors])
    k = D.shape[1]
    w = minimize(lambda w: ((y[:n_pre] - D[:n_pre] @ w) ** 2).sum(), np.ones(k) / k, method="SLSQP",
                 bounds=[(0, 1)] * k, constraints={"type": "eq", "fun": lambda w: w.sum() - 1}).x
    gap = y - D @ w
    return panel.North.to_numpy() - gap, pd.Series(w, donors.columns)   # counterfactual, weights


cf_trend, w_trend = sc_with_trend(panel2, N_PRE)
att_trend = (panel2.North - cf_trend).iloc[N_PRE:].mean()
plot_sc(panel2, cf_trend, "HULL_SEED world: SC with a trend")

cf_trend_fake, _ = sc_with_trend(panel2.iloc[:N_PRE], fake)
gap_fake = (panel2.North.iloc[:N_PRE] - cf_trend_fake).iloc[fake:].mean()
plot_sc(panel2.iloc[:N_PRE], cf_trend_fake, "HULL_SEED world: SC with a trend, fake rollout date", fake=fake)

In [ ]:
tr2 = truth2["att_north_store_abs"]
pd.DataFrame({
    "ATT (k$)": [tr2, sc2.att, sdid2.effects.att, att_trend],
    "error": [0, sc2.att / tr2 - 1, sdid2.effects.att / tr2 - 1, att_trend / tr2 - 1],
    "fake-date gap (% of true effect)": [np.nan, sc2_fake.att / tr2, np.nan, gap_fake / tr2],
}, index=["truth", "classic SC", "SDID", "SC with a trend"]).style.format(
    {"ATT (k$)": "{:.2f}", "error": "{:+.0%}", "fake-date gap (% of true effect)": "{:+.0%}"}, na_rep="-")

**Takeaways**

- Check that the treated unit is inside the donors' range (level *and* growth) before you trust SC.
- A good-looking pre-period fit isn't enough. Look at the gap, and run the fake-date check.
- When the treated unit is outside the hull, switch to an estimator whose assumptions fit: here, one that allows a separate trend. Say which assumption you swapped in.